# StoCFL: efeito da fração de clientes por rodada (m)

Pipeline de teste da **clusterização** do StoCFL (`StoCFL-FCC+CFL.py --train 0`) em Rotated MNIST, variando a fração `m` de clientes sorteados por rodada (**100%, 50%, 25%, 10% e 5%**) e o nº `k` de grupos reais (**4, 8 e 16**).

- **Cenário:** Rotated MNIST, `n = 240` clientes e `k` grupos reais em `KS`. Os grupos são rotações em `k` ângulos igualmente espaçados (`k = 4`: 0°, 90°, 180°, 270°; `k = 8`: 0°, 45°, ...; `k = 16`: 0°, 22,5°, ...), e o cliente `c` pertence ao grupo `c * k // n`. `n = 240` é divisível por 4, 8 e 16.
- **Sem treino:** só a observação de `Ψ` e a fusão de clusters. O que muda com `m` é quantos clientes são observados e quantos clusters entram na matriz de cosseno a cada rodada.
- **Rodadas:** no máximo 50.
- **Como roda:** cada combinação de `k` e `m` chama o script por `subprocess`, sem alterar o código do StoCFL. O notebook lê o `cluster.pkl` e o log de cada execução.
- **Ambiente:** o kernel é o `.venv` da raiz (ver o README da raiz). Na primeira execução, o MNIST é baixado e as partições rotacionadas são gravadas em `datasets/rotated_mnist_240_<seed>/` (`k = 4`) e `datasets/rotated_mnist_240_<seed>_k<k>/` (outros `k`).

Métricas por rodada, contra o grupo real de cada cliente:

| Métrica | Significado |
|---|---|
| nº de clusters | tende a `k` quando `τ` está bem calibrado |
| ARI | 1 = partição igual à real |
| pureza | fração de clientes cujo cluster é dominado pelo grupo deles |
| cobertura | fração dos clientes já sorteados (e, portanto, com `Ψ`) |

In [1]:
import re
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots
from sklearn.metrics import adjusted_rand_score

PROJETO = Path.cwd()
SCRIPT = "StoCFL-FCC+CFL.py"
assert (PROJETO / SCRIPT).exists(), "abra o notebook a partir de projetos/StoCFL/"

# Parâmetros do teste
N = 240                           # nº de clientes (divisível por todo k de KS)
KS = [4, 8, 16]                   # k: grupos reais do Rotated MNIST, rotações em k ângulos igualmente espaçados (4: 0°, 90°, ...)
ROUNDS = 50                       # nº de rodadas de comunicação (no máximo 50)
TAU = 0.9                         # limiar de cosseno para fundir clusters (--tau)
MU = 0.05                         # só entra no nome da pasta de logs: sem treino, não afeta a clusterização
SEEDS = [0]                       # cada seed gera uma nova partição dos dados (~750 MB em disco)
FRACOES = [1.0, 0.5, 0.25, 0.1, 0.05]   # m: fração dos clientes sorteada por rodada
REEXECUTAR = False                # True ignora execuções já concluídas e roda de novo

assert ROUNDS <= 50 and all(N % k == 0 for k in KS)


def verdade(k):
    # gabarito: o cliente c pertence ao grupo c * k // N (blocos contíguos de N/k clientes)
    return [c * k // N for c in range(N)]


# o script só acrescenta o k ao nome das pastas quando ele não é o padrão (4)
K_PADRAO = 4
print("GPU:", torch.cuda.is_available(), "| clientes por rodada:", {f: int(N * f) for f in FRACOES})

/home/luiz/cfl_stocfl/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:70: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


GPU: True | clientes por rodada: {1.0: 240, 0.5: 120, 0.25: 60, 0.1: 24, 0.05: 12}


## 1. Execução

`executar(k, frac, seed)` roda o script uma vez. Para cada `k` em `KS`, o script gera as `N` partições com `k` rotações (`--k`). Se já existir uma execução concluída com os mesmos parâmetros (com `ROUNDS` entradas em `cluster.pkl`), ela é reaproveitada, a menos que `REEXECUTAR = True`.

Atenção: o script grava `psi.pkl` a cada rodada (`N` vetores de ~6,5 MB, ou seja, ~1,6 GB por gravação com `N = 240`), então cada execução leva alguns minutos e usa espaço em `logs/`.

In [2]:
def sufixo(k, sep):
    # o script só acrescenta o k aos nomes quando ele não é o padrão
    return "" if k == K_PADRAO else f"{sep}k{k}"


def dados_prontos(k, seed):
    # as partições rotacionadas já foram gravadas? (decide o --process do script)
    d = PROJETO / f"datasets/rotated_mnist_{N}_{seed}{sufixo(k, '_')}"
    return len(list((d / "train").glob("data*.pkl"))) == N and len(list((d / "test").glob("data*.pkl"))) == k


def pastas_de_log(k, m, seed):
    # o nome da pasta de logs do script traz n, m, seed, mu e k (se não for o padrão), mas não o tau; ordena da mais antiga para a mais nova
    padrao = f"runs-rotated-mnist-n{N}-p{m}-seed{seed}-lambda{MU}{sufixo(k, '-')}-time-*"
    return sorted((PROJETO / "logs").glob(padrao), key=lambda p: p.stat().st_mtime)


def concluida(pasta):
    # execução completa = cluster.pkl tem uma partição por rodada
    try:
        return len(torch.load(pasta / "cluster.pkl", weights_only=False)) == ROUNDS
    except Exception:
        return False


def executar(k, frac, seed):
    m = int(N * frac)
    if not REEXECUTAR:   # reaproveita a execução mais recente que terminou
        for pasta in reversed(pastas_de_log(k, m, seed)):
            if concluida(pasta):
                print(f"k={k} m={frac:.0%} seed={seed}: reaproveitando {pasta.name}")
                return pasta, None
    cmd = [sys.executable, SCRIPT, "--n", str(N), "--com_round", str(ROUNDS),
           "--num_per_round", str(m), "--mu", str(MU), "--tau", str(TAU),
           "--train", "0", "--seed", str(seed), "--dataset", "mnist", "--k", str(k),
           "--process", "0" if dados_prontos(k, seed) else "1"]
    (PROJETO / "logs").mkdir(exist_ok=True)
    saida = PROJETO / "logs" / f"pipeline-k{k}-m{m}-seed{seed}.txt"
    inicio = time.time()
    with open(saida, "w") as f:
        r = subprocess.run(cmd, cwd=PROJETO, stdout=f, stderr=subprocess.STDOUT)
    duracao = time.time() - inicio
    if r.returncode != 0:
        print(saida.read_text()[-3000:])
        raise RuntimeError(f"falhou: {' '.join(cmd)}")
    # a pasta criada por esta execução é a mais nova com mtime a partir do início
    pasta = next(p for p in reversed(pastas_de_log(k, m, seed)) if p.stat().st_mtime >= inicio - 1)
    print(f"k={k} m={frac:.0%} seed={seed}: {duracao:.0f} s -> {pasta.name}")
    return pasta, duracao


execucoes = {}
for k in KS:
    for seed in SEEDS:
        for frac in FRACOES:
            execucoes[(k, frac, seed)] = executar(k, frac, seed)

k=4 m=100% seed=0: reaproveitando runs-rotated-mnist-n240-p240-seed0-lambda0.05-time-11-06-11:48
k=4 m=50% seed=0: reaproveitando runs-rotated-mnist-n240-p120-seed0-lambda0.05-time-11-06-11:49
k=4 m=25% seed=0: reaproveitando runs-rotated-mnist-n240-p60-seed0-lambda0.05-time-11-06-11:50
k=4 m=10% seed=0: reaproveitando runs-rotated-mnist-n240-p24-seed0-lambda0.05-time-11-06-11:52
k=4 m=5% seed=0: 42 s -> runs-rotated-mnist-n240-p12-seed0-lambda0.05-time-11-06-11:53
k=8 m=100% seed=0: 114 s -> runs-rotated-mnist-n240-p240-seed0-lambda0.05-k8-time-11-06-11:54
k=8 m=50% seed=0: 78 s -> runs-rotated-mnist-n240-p120-seed0-lambda0.05-k8-time-11-06-11:55
k=8 m=25% seed=0: 92 s -> runs-rotated-mnist-n240-p60-seed0-lambda0.05-k8-time-11-06-11:57
k=8 m=10% seed=0: 54 s -> runs-rotated-mnist-n240-p24-seed0-lambda0.05-k8-time-11-06-11:58
k=8 m=5% seed=0: 48 s -> runs-rotated-mnist-n240-p12-seed0-lambda0.05-k8-time-11-06-11:59
k=16 m=100% seed=0: 148 s -> runs-rotated-mnist-n240-p240-seed0-lambda0.

## 2. Métricas por rodada

O `cluster.pkl` guarda, para cada rodada, o dicionário `cluster_id → membros` (clusters mortos ficam com lista vazia). A cobertura vem das linhas `client id this round` do log principal.

In [ ]:
def rotulos(particao):
    # dicionário cluster -> membros vira lista cliente -> cluster (formato do adjusted_rand_score)
    r = [None] * N
    for cid, membros in particao.items():
        for c in membros:
            r[c] = cid
    return r


def pureza(particao, gabarito):
    # fração de clientes que pertencem ao grupo real dominante do seu cluster
    ok = 0
    for membros in particao.values():
        if membros:
            ok += np.bincount([gabarito[c] for c in membros]).max()
    return ok / N


def misturados(particao, gabarito):
    # nº de clusters com clientes de mais de um grupo real (fusões erradas, irreversíveis)
    return sum(1 for membros in particao.values() if len({gabarito[c] for c in membros}) > 1)


def cobertura(pasta):
    # fração acumulada de clientes já sorteados por rodada, lida das linhas "client id this round" do log
    visto, serie = set(), []
    for linha in (pasta / "rotated_mnist.log").read_text().splitlines():
        achou = re.search(r"client id this round \[(.*?)\]", linha)
        if achou:
            visto /home/luiz/cfl_stocfl/projetos/StoCFL/teste-fracao-clientes.ipynb|= {int(x) for x in achou.group(1).split(",")}
            serie.append(len(visto) / N)
    return serie


def metricas(pasta, k):
    # uma linha por rodada; clusters mortos (lista vazia) não contam
    gabarito = verdade(k)
    log = torch.load(pasta / "cluster.pkl", weights_only=False)
    cob = cobertura(pasta)
    linhas = []
    for r, particao in enumerate(log):
        linhas.append({
            "rodada": r + 1,
            "clusters": sum(1 for v in particao.values() if v),
            "ari": adjusted_rand_score(gabarito, rotulos(particao)),
            "pureza": pureza(particao, gabarito),
            "misturados": misturados(particao, gabarito),
            "cobertura": cob[r],
        })
    return pd.DataFrame(linhas), log


def rodada_estavel(log):
    """Primeira rodada (1-based) a partir da qual a partição não muda mais."""
    chave = lambda p: frozenset(frozenset(v) for v in p.values() if v)
    final = chave(log[-1])
    r = len(log)
    while r > 1 and chave(log[r - 2]) == final:
        r -= 1
    return r


# resumo da última rodada de cada execução
tabelas, resumo = {}, []
for (k, frac, seed), (pasta, duracao) in execucoes.items():
    df, log = metricas(pasta, k)
    df.insert(0, "seed", seed)
    df.insert(0, "m", frac)
    df.insert(0, "k", k)
    tabelas[(k, frac, seed)] = df
    ultimo = df.iloc[-1]
    atingiu = df.index[df["clusters"] == k]   # rodadas em que o nº de clusters chegou a k
    resumo.append({
        "k": k, "m": f"{frac:.0%}", "clientes/rodada": int(N * frac), "seed": seed,
        "clusters finais": int(ultimo["clusters"]),
        "1ª rodada com clusters=k": int(df.loc[atingiu[0], "rodada"]) if len(atingiu) else None,
        "rodada estável": rodada_estavel(log),
        "ARI final": round(ultimo["ari"], 3),
        "pureza final": round(ultimo["pureza"], 3),
        "clusters mistos": int(ultimo["misturados"]),
        "cobertura final": round(ultimo["cobertura"], 2),
        "tempo (s)": None if duracao is None else round(duracao),
    })

resumo = pd.DataFrame(resumo).astype({"1ª rodada com clusters=k": "Int64", "tempo (s)": "Int64"})
resumo

,k,m,clientes/rodada,seed,clusters finais,1ª rodada com clusters=k,rodada estável,ARI final,pureza final,clusters mistos,cobertura final,tempo (s)
0,4,100%,240,0,4,1,1,1.000,1.0,0,1.00,<NA>
1,4,50%,120,0,4,10,10,1.000,1.0,0,1.00,<NA>
2,4,25%,60,0,4,20,20,1.000,1.0,0,1.00,<NA>
3,4,10%,24,0,6,<NA>,49,0.989,1.0,0,0.99,<NA>
4,4,5%,12,0,35,<NA>,50,0.820,1.0,0,0.89,42
5,8,100%,240,0,8,1,1,1.000,1.0,0,1.00,114
6,8,50%,120,0,8,10,10,1.000,1.0,0,1.00,78
7,8,25%,60,0,8,20,20,1.000,1.0,0,1.00,92
8,8,10%,24,0,10,<NA>,49,0.990,1.0,0,0.99,54
9,8,5%,12,0,56,<NA>,50,0.737,1.0,0,0.89,48


## 3. Gráficos

Gráficos interativos em Plotly: passe o mouse para ver os valores por rodada e clique na legenda para ocultar um valor de `m`. Um painel por métrica e uma figura por `k` de `KS`. Para mais de uma seed, a linha é a média e a faixa vai do mínimo ao máximo.

In [4]:
CORES = {1.0: "#2a78d6", 0.5: "#eb6834", 0.25: "#1baf7a", 0.1: "#eda100", 0.05: "#8a5cd0"}
TRACOS = {1.0: ("solid", "circle"), 0.5: ("dash", "square"), 0.25: ("dashdot", "triangle-up"), 0.1: ("dot", "diamond"), 0.05: ("longdash", "x")}

# um painel por métrica, uma linha por m, uma figura por k
todas_k = pd.concat(tabelas.values(), ignore_index=True)
paineis = [("clusters", "Nº de clusters"), ("ari", "ARI contra os grupos reais"), ("cobertura", "Clientes já observados")]

for k in KS:
    todas = todas_k[todas_k["k"] == k]
    fig = make_subplots(rows=1, cols=3, subplot_titles=[t for _, t in paineis], horizontal_spacing=0.07)
    for i, (col, _) in enumerate(paineis, start=1):
        for frac in FRACOES:
            g = todas[todas["m"] == frac].groupby("rodada")[col]
            media = g.mean()
            dash, simbolo = TRACOS.get(frac, ("solid", "circle"))
            nome = f"m = {frac:.0%}"
            if len(SEEDS) > 1:  # faixa mín-máx entre as seeds
                fig.add_trace(go.Scatter(x=list(media.index) + list(media.index[::-1]),
                                         y=list(g.max()) + list(g.min()[::-1]), fill="toself",
                                         fillcolor=CORES.get(frac), opacity=0.15, line=dict(width=0),
                                         hoverinfo="skip", showlegend=False, legendgroup=nome), row=1, col=i)
            fig.add_trace(go.Scatter(x=media.index, y=media.values, mode="lines+markers", name=nome,
                                     legendgroup=nome, showlegend=(i == 1),
                                     line=dict(color=CORES.get(frac), dash=dash, width=2),
                                     marker=dict(symbol=simbolo, size=6, color=CORES.get(frac)),
                                     hovertemplate="rodada %{x}: %{y:.3g}<extra>" + nome + "</extra>"), row=1, col=i)
        fig.update_xaxes(title_text="rodada", row=1, col=i)
    fig.add_hline(y=k, line=dict(color="gray", width=1), annotation_text=f"k real = {k}", annotation_position="top right", row=1, col=1)
    fig.update_layout(title=f"k = {k} grupos reais", height=460, hovermode="x unified", margin=dict(t=80, b=40, l=50, r=20),
                      legend=dict(orientation="h", y=-0.2), template="simple_white")
    fig.update_xaxes(showgrid=True, gridcolor="rgba(128,128,128,0.2)")
    fig.update_yaxes(showgrid=True, gridcolor="rgba(128,128,128,0.2)")
    fig.show()

## 4. Leitura

Rode as células acima e compare a tabela do item 2 (uma linha por `k` e `m`):

- **`rodada estável`** menor = a partição para de mudar mais cedo.
- **`clusters finais` ≠ `k`** com `ARI` baixo indica que `τ` (`TAU`, 0.9 por padrão) não separou os grupos naquele `m`, ou que a clusterização não terminou nas 50 rodadas.
- **`cobertura`** mostra quanto da lentidão vem de clientes ainda não observados (o cliente só entra na fusão depois de sorteado).
- Com `m = 100%` todos são observados na rodada 1 e todos os clusters entram na matriz de cosseno a cada rodada, então é o limite mais rápido que o algoritmo atual permite.
- Com uma só seed, as diferenças pequenas entre os `m` podem ser ruído da amostragem. Para conclusões mais firmes, use `SEEDS = [0, 42, 1998]` (cada seed gera e grava uma nova partição dos dados).

## 5. Busca de τ

Como os clientes se agrupam em função de `τ`? O gabarito é o cenário ideal: os `k` grupos reais (`verdade(k)`). Para cada `k` em `TAU_KS` e cada `m` em `TAU_FRACOES`:

1. **Varredura grossa** em `TAU_GRADE` (de 0.05 a 1.0). Com `τ = 1.0` nenhum par se funde (cosseno > 1 é impossível), então é o extremo "cada cliente é um cluster".
2. **Bisseção:** o melhor resultado é o `τ` de maior ARI final contra o gabarito (se vários empatam, o intervalo `[τ_min, τ_max]` deles). A cada passo, avalia-se o ponto médio do maior intervalo ainda não resolvido entre a borda desse conjunto ótimo e o vizinho avaliado. Para depois de `TAU_BISSECCOES` passos ou quando o intervalo fica menor que `TAU_LARGURA_MIN`.
3. **τ sugerido:** o centro do intervalo ótimo, que fica mais longe das duas bordas onde a partição piora.

Cada execução é uma chamada do script com `--tau` diferente; o resultado vai para `logs/busca-tau.json`, então reexecutar a célula reaproveita o que já foi rodado (a menos que `REEXECUTAR = True`). Só a primeira seed (`SEEDS[0]`) é usada. Atenção ao espaço em disco: cada execução grava ~1,6 GB em `logs/`, e a busca faz até 11 execuções por par `(k, m)` (com `TAU_KS = [4, 8, 16]` e 3 valores de `m`, até 99 execuções, ~160 GB). Reduza `TAU_KS` ou `TAU_FRACOES` se faltar espaço.

In [5]:
import json

# Parâmetros da busca de τ
TAU_GRADE = [0.05, 0.25, 0.5, 0.75, 1.0]   # varredura grossa, com os extremos 0.05 e 1.0
TAU_KS = list(KS)                          # k: uma busca separada para cada nº de grupos reais
TAU_FRACOES = [1.0, 0.25, 0.1]             # m: uma busca separada para cada fração
TAU_BISSECCOES = 6                         # máximo de pontos avaliados depois da varredura
TAU_LARGURA_MIN = 0.005                    # para quando o maior intervalo aberto fica menor que isso
TAU_SEED = SEEDS[0]                        # só uma seed na busca
# o nome da pasta de logs não traz o tau, então guardamos aqui qual pasta é de qual (n, k, m, seed, tau)
CACHE_TAU = PROJETO / "logs" / "busca-tau.json"


def carregar_cache():
    try:
        return json.loads(CACHE_TAU.read_text())
    except Exception:
        return {}


def executar_tau(k, frac, seed, tau):
    # igual a executar(), mas com --tau variável e cache próprio
    m = int(N * frac)
    chave = f"{N}|{k}|{m}|{seed}|{tau:.4f}"
    cache = carregar_cache()
    if chave in cache and not REEXECUTAR and concluida(PROJETO / "logs" / cache[chave]):
        return PROJETO / "logs" / cache[chave]
    # o script nomeia a pasta só até o minuto: espera virar o minuto para não cair na pasta de outra execução
    while any(p.name.endswith(time.strftime("%m-%d-%H:%M")) for p in pastas_de_log(k, m, seed)):
        time.sleep(5)
    cmd = [sys.executable, SCRIPT, "--n", str(N), "--com_round", str(ROUNDS),
           "--num_per_round", str(m), "--mu", str(MU), "--tau", str(tau),
           "--train", "0", "--seed", str(seed), "--dataset", "mnist", "--k", str(k),
           "--process", "0" if dados_prontos(k, seed) else "1"]
    saida = PROJETO / "logs" / f"pipeline-k{k}-m{m}-seed{seed}-tau{tau:.4f}.txt"
    inicio = time.time()
    with open(saida, "w") as f:
        r = subprocess.run(cmd, cwd=PROJETO, stdout=f, stderr=subprocess.STDOUT)
    if r.returncode != 0:
        print(saida.read_text()[-3000:])
        raise RuntimeError(f"falhou: {' '.join(cmd)}")
    pasta = next(p for p in reversed(pastas_de_log(k, m, seed)) if p.stat().st_mtime >= inicio - 1)
    cache = carregar_cache()
    cache[chave] = pasta.name
    CACHE_TAU.write_text(json.dumps(cache, indent=1))
    return pasta


def avaliar_tau(k, frac, tau):
    # métricas da última rodada: é a partição final que importa na busca
    pasta = executar_tau(k, frac, TAU_SEED, tau)
    df, log = metricas(pasta, k)
    u = df.iloc[-1]
    return {"k": k, "m": frac, "tau": tau, "clusters": int(u["clusters"]), "ari": float(u["ari"]),
            "pureza": float(u["pureza"]), "misturados": int(u["misturados"]),
            "rodada estável": rodada_estavel(log), "pasta": pasta.name}


def buscar_tau(k, frac):
    # varredura grossa e depois bisseção em torno do(s) τ de maior ARI contra o gabarito
    res = {}   # tau -> métricas

    def ver(tau, fase):
        # avalia o tau (se ainda não foi avaliado) e imprime o resultado
        tau = round(tau, 4)
        if tau not in res:
            res[tau] = {**avaliar_tau(k, frac, tau), "fase": fase}
            r = res[tau]
            print(f"k={k} m={frac:.0%} τ={tau:<7} {fase:<9} clusters={r['clusters']:<4} ARI={r['ari']:.3f} pureza={r['pureza']:.3f}")

    for t in TAU_GRADE:
        ver(t, "grade")
    for _ in range(TAU_BISSECCOES):
        taus = sorted(res)
        melhor = max(r["ari"] for r in res.values())
        otimos = [t for t in taus if res[t]["ari"] >= melhor - 1e-9]   # empates em ARI formam um intervalo
        i_lo, i_hi = taus.index(otimos[0]), taus.index(otimos[-1])
        abertos = []   # intervalos entre a borda do conjunto ótimo e o vizinho avaliado
        if i_lo > 0:
            abertos.append((taus[i_lo - 1], taus[i_lo]))
        if i_hi < len(taus) - 1:
            abertos.append((taus[i_hi], taus[i_hi + 1]))
        if not abertos:
            break
        a, b = max(abertos, key=lambda x: x[1] - x[0])   # refina primeiro o maior intervalo, onde a borda é mais incerta
        if b - a < TAU_LARGURA_MIN:
            break
        ver((a + b) / 2, "bisseção")
    return pd.DataFrame(res.values()).sort_values("tau").reset_index(drop=True)


# resultado: um DataFrame por (k, m), ordenado por tau
busca = {(k, frac): buscar_tau(k, frac) for k in TAU_KS for frac in TAU_FRACOES}

k=4 m=100% τ=0.05    grade     clusters=1    ARI=0.000 pureza=0.250
k=4 m=100% τ=0.25    grade     clusters=1    ARI=0.000 pureza=0.250
k=4 m=100% τ=0.5     grade     clusters=4    ARI=1.000 pureza=1.000
k=4 m=100% τ=0.75    grade     clusters=4    ARI=1.000 pureza=1.000
k=4 m=100% τ=1.0     grade     clusters=240  ARI=0.000 pureza=1.000
k=4 m=100% τ=0.375   bisseção  clusters=4    ARI=1.000 pureza=1.000
k=4 m=100% τ=0.875   bisseção  clusters=4    ARI=1.000 pureza=1.000
k=4 m=100% τ=0.3125  bisseção  clusters=2    ARI=0.497 pureza=0.500
k=4 m=100% τ=0.9375  bisseção  clusters=12   ARI=0.955 pureza=1.000
k=4 m=100% τ=0.3438  bisseção  clusters=4    ARI=1.000 pureza=1.000
k=4 m=100% τ=0.9062  bisseção  clusters=4    ARI=1.000 pureza=1.000
k=4 m=25% τ=0.05    grade     clusters=1    ARI=0.000 pureza=0.250
k=4 m=25% τ=0.25    grade     clusters=1    ARI=0.000 pureza=0.250
k=4 m=25% τ=0.5     grade     clusters=4    ARI=1.000 pureza=1.000
k=4 m=25% τ=0.75    grade     clusters=4    ARI=1.0

### Resultados da busca

Para cada `k` e `m`: o intervalo de `τ` com o maior ARI, o `τ` sugerido (centro do intervalo) e a incerteza das bordas (o intervalo até o vizinho avaliado, onde a borda real está).

Nos gráficos: o painel de clusters usa escala log (de 1 a 200 clusters), e as faixas coloridas de fundo marcam o intervalo de `τ` com o maior ARI de cada `m`. As curvas de `m` = 100% e 25% coincidem em ARI e pureza, por isso os marcadores têm tamanhos diferentes.

In [6]:
# Para cada (k, m): intervalo de τ com o maior ARI, τ sugerido (centro) e incerteza das bordas
linhas = []
for (k, frac), d in busca.items():
    melhor = d["ari"].max()
    otimos = d[d["ari"] >= melhor - 1e-9]
    lo, hi = otimos["tau"].min(), otimos["tau"].max()
    taus = list(d["tau"])
    # vizinhos avaliados fora do intervalo ótimo: a borda real está entre eles e a borda do ótimo
    viz_lo = taus[taus.index(lo) - 1] if taus.index(lo) > 0 else None
    viz_hi = taus[taus.index(hi) + 1] if taus.index(hi) < len(taus) - 1 else None
    linhas.append({
        "k": k, "m": f"{frac:.0%}", "execuções": len(d), "ARI máx": round(melhor, 3),
        "clusters no ótimo": sorted(set(otimos["clusters"])),
        "τ ótimo (mín)": lo, "τ ótimo (máx)": hi, "τ sugerido": round((lo + hi) / 2, 4),
        "borda inferior em": None if viz_lo is None else f"({viz_lo}, {lo}]",
        "borda superior em": None if viz_hi is None else f"[{hi}, {viz_hi})",
    })
display(pd.DataFrame(linhas))
# todas as execuções de cada (k, m), em ordem de tau
for (k, frac), d in busca.items():
    print(f"\nk = {k}, m = {frac:.0%}")
    display(d.drop(columns=["k", "m", "pasta"]).round(3))

,k,m,execuções,ARI máx,clusters no ótimo,τ ótimo (mín),τ ótimo (máx),τ sugerido,borda inferior em,borda superior em
0,4,100%,11,1.000,[4],0.3438,0.9062,0.6250,"(0.3125, 0.3438]","[0.9062, 0.9375)"
1,4,25%,11,1.000,[4],0.3438,0.9062,0.6250,"(0.3125, 0.3438]","[0.9062, 0.9375)"
2,4,10%,11,0.989,[6],0.3438,0.9062,0.6250,"(0.3125, 0.3438]","[0.9062, 0.9375)"
3,8,100%,11,1.000,[8],0.4688,0.9375,0.7031,"(0.4375, 0.4688]","[0.9375, 0.9688)"
4,8,25%,11,1.000,[8],0.4688,0.9375,0.7031,"(0.4375, 0.4688]","[0.9375, 0.9688)"
5,8,10%,11,0.990,[10],0.4375,0.9375,0.6875,"(0.4062, 0.4375]","[0.9375, 0.9688)"
6,16,100%,11,1.000,[16],0.7500,0.9688,0.8594,"(0.7188, 0.75]","[0.9688, 1.0)"
7,16,25%,11,1.000,[16],0.7500,0.9688,0.8594,"(0.7188, 0.75]","[0.9688, 1.0)"
8,16,10%,11,0.965,[22],0.7500,0.9375,0.8438,"(0.7188, 0.75]","[0.9375, 0.9688)"



k = 4, m = 100%


,tau,clusters,ari,pureza,misturados,rodada estável,fase
0,0.050,1,0.000,0.25,1,1,grade
1,0.250,1,0.000,0.25,1,1,grade
2,0.312,2,0.497,0.50,2,1,bisseção
3,0.344,4,1.000,1.00,0,1,bisseção
4,0.375,4,1.000,1.00,0,1,bisseção
5,0.500,4,1.000,1.00,0,1,grade
6,0.750,4,1.000,1.00,0,1,grade
7,0.875,4,1.000,1.00,0,1,bisseção
8,0.906,4,1.000,1.00,0,1,bisseção
9,0.938,12,0.955,1.00,0,2,bisseção



k = 4, m = 25%


,tau,clusters,ari,pureza,misturados,rodada estável,fase
0,0.050,1,0.000,0.25,1,20,grade
1,0.250,1,0.000,0.25,1,20,grade
2,0.312,3,0.712,0.75,1,20,bisseção
3,0.344,4,1.000,1.00,0,20,bisseção
4,0.375,4,1.000,1.00,0,20,bisseção
5,0.500,4,1.000,1.00,0,20,grade
6,0.750,4,1.000,1.00,0,20,grade
7,0.875,4,1.000,1.00,0,20,bisseção
8,0.906,4,1.000,1.00,0,20,bisseção
9,0.938,16,0.933,1.00,0,20,bisseção



k = 4, m = 10%


... (9 outputs left out)


k = 16, m = 25%


,tau,clusters,ari,pureza,misturados,rodada estável,fase
0,0.050,1,0.000,0.062,1,20,grade
1,0.250,1,0.000,0.062,1,20,grade
2,0.500,1,0.000,0.062,1,20,grade
3,0.625,5,0.001,0.100,4,20,bisseção
4,0.688,8,0.017,0.167,6,38,bisseção
5,0.719,10,0.273,0.379,3,37,bisseção
6,0.750,16,1.000,1.000,0,20,grade
7,0.875,16,1.000,1.000,0,20,bisseção
8,0.938,16,1.000,1.000,0,20,bisseção
9,0.969,16,1.000,1.000,0,20,bisseção



k = 16, m = 10%


,tau,clusters,ari,pureza,misturados,rodada estável,fase
0,0.050,3,-0.000,0.071,1,49,grade
1,0.250,3,-0.000,0.071,1,49,grade
2,0.500,3,-0.000,0.071,1,49,grade
3,0.625,22,0.003,0.192,5,49,bisseção
4,0.688,14,0.035,0.217,6,49,bisseção
5,0.719,13,0.518,0.542,5,50,bisseção
6,0.750,22,0.965,1.000,0,50,grade
7,0.875,22,0.965,1.000,0,50,bisseção
8,0.938,22,0.965,1.000,0,50,bisseção
9,0.969,23,0.961,1.000,0,50,bisseção


In [7]:
# mesmos 3 painéis do item 3, mas em função de τ e não da rodada, uma figura por k; losango = ponto da bisseção
# o nº de clusters vai de 1 a N (τ = 1.0: cada cliente é um cluster), então esse painel usa escala log
paineis = [("clusters", "Nº de clusters finais (escala log)"), ("ari", "ARI final contra os grupos reais"), ("pureza", "Pureza final")]
TAMANHOS = {1.0: 12, 0.25: 8, 0.1: 5}   # marcadores de tamanhos diferentes: curvas de m diferentes podem coincidir e se sobrepor
for k in TAU_KS:
    busca_k = {frac: d for (kk, frac), d in busca.items() if kk == k}
    fig = make_subplots(rows=1, cols=3, subplot_titles=[t for _, t in paineis], horizontal_spacing=0.08)
    for i, (col, _) in enumerate(paineis, start=1):
        for frac, d in busca_k.items():
            d = d.sort_values("tau")
            dash, _ = TRACOS.get(frac, ("solid", "circle"))
            nome = f"m = {frac:.0%}"
            fig.add_trace(go.Scatter(
                x=d["tau"], y=d[col], mode="lines+markers", name=nome, legendgroup=nome, showlegend=(i == 1),
                line=dict(color=CORES.get(frac), dash=dash, width=2),
                marker=dict(symbol=[("diamond" if f == "bisseção" else "circle") for f in d["fase"]],
                            size=TAMANHOS.get(frac, 8), color=CORES.get(frac), opacity=0.85,
                            line=dict(color="white", width=1)),
                hovertemplate="τ=%{x}: %{y:.3g}<extra>" + nome + "</extra>"), row=1, col=i)
        fig.update_xaxes(title_text="τ (losango = bisseção)", range=[0, 1.04], dtick=0.25, row=1, col=i)
    # faixa de fundo: intervalo de τ com o maior ARI de cada m
    for frac, d in busca_k.items():
        otimos = d[d["ari"] >= d["ari"].max() - 1e-9]
        for i in (1, 2, 3):
            fig.add_vrect(x0=otimos["tau"].min(), x1=otimos["tau"].max(), fillcolor=CORES.get(frac), opacity=0.05,
                          line_width=0, layer="below", row=1, col=i)
    # linha de referência: k real
    fig.add_hline(y=k, line=dict(color="gray", width=1, dash="dot"), annotation_text=f"k real = {k}",
                  annotation_position="top left", row=1, col=1)
    ticks = sorted({1, 2, 4, 10, 20, 50, 100, N, k})
    fig.update_yaxes(type="log", tickvals=ticks, title_text="clusters", row=1, col=1)
    fig.update_yaxes(range=[-0.05, 1.05], title_text="ARI", row=1, col=2)
    fig.update_yaxes(range=[0.0, 1.05], title_text="pureza", row=1, col=3)
    fig.update_layout(title=f"k = {k} grupos reais", height=460, hovermode="x unified",
                      margin=dict(t=80, b=40, l=60, r=20), legend=dict(orientation="h", y=-0.22), template="simple_white")
    fig.update_xaxes(showgrid=True, gridcolor="rgba(128,128,128,0.2)")
    fig.update_yaxes(showgrid=True, gridcolor="rgba(128,128,128,0.2)")
    fig.show()